In [1]:
!pip install -q transformers==4.46.1 IndicTransToolkit sentencepiece
print("Install finished")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 42.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 548.4/548.4 kB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 46.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.8/53.8 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 867.8/867.8 kB 24.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.6/129.6 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.7/7.7 MB 31.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.1/121.1 kB 5.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is t

In [5]:
from google.colab import userdata
from huggingface_hub import login

login(token=userdata.get("HF_TOKEN"))
print("Logged in to Hugging Face")

Logged in to Hugging Face


In [14]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "facebook/nllb-200-distilled-600M"
tokenizer = AutoTokenizer.from_pretrained(model_name, src_lang="eng_Latn")
model = AutoModelForSeq2SeqLM.from_pretrained(
    model_name, torch_dtype=torch.float16
).to("cuda")
print("Translation model loaded")

tokenizer_config.json:   0%|          | 0.00/564 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/4.85M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.3M [00:00<?, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/846 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/2.46G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

Translation model loaded


In [15]:
def translate_from_english(sentences, tgt_lang):
    inputs = tokenizer(
        sentences, padding=True, truncation=True, max_length=256, return_tensors="pt"
    ).to("cuda")

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            forced_bos_token_id=tokenizer.convert_tokens_to_ids(tgt_lang),
            num_beams=5,
            max_length=256,
        )
    return tokenizer.batch_decode(outputs, skip_special_tokens=True)

In [16]:
import time

sentences = [
    "Can you hear my voice?",
    "I don't know where to go. Please help me.",
]

languages = [("hin_Deva", "Hindi"), ("tam_Taml", "Tamil"), ("tel_Telu", "Telugu")]

for code, name in languages:
    start = time.time()
    results = translate_from_english(sentences, code)
    print(f"--- {name} ({time.time() - start:.2f}s) ---")
    for r in results:
        print(r)

--- Hindi (1.20s) ---
क्या आप मेरी आवाज सुन सकते हैं?
मुझे नहीं पता कि कहाँ जाना है, कृपया मेरी मदद करें।
--- Tamil (0.31s) ---
நீங்கள் என் குரல் கேட்க முடியும்?
நான் எங்கே செல்ல வேண்டும் என்று எனக்கு தெரியாது. தயவு செய்து எனக்கு உதவுங்கள்.
--- Telugu (0.29s) ---
మీరు నా వాయిస్ వినవచ్చు?
నేను ఎక్కడికి వెళ్ళాలో నాకు తెలియదు. దయచేసి నాకు సహాయం చేయండి.
